# Declared Workflows (Offline)

Plan and run two workflows with local models and tensors. This notebook runs in CI.

In [ ]:
import torch
from torch import nn
from tensordict import TensorDict

torch.manual_seed(0)


class TinyConceptModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(10, 10)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(10, 2)

    def forward(self, input):
        return self.linear2(self.relu(self.linear1(input)))


class FeatureMap(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(10, 12)

    def forward(self, input):
        return self.linear(input).relu().reshape(-1, 3, 2, 2)


class TinyFeatureModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = FeatureMap()
        self.head = nn.Linear(12, 2)

    def forward(self, input):
        features = self.features(input)
        return self.head(features.flatten(1))


concept_model = TinyConceptModel().eval()
dimension_model = TinyFeatureModel().eval()
examples = torch.randn(6, 10)
labels = torch.tensor([1, 1, 1, 0, 0, 0])

## Concept-conditioned attribution: two model passes

The first LRP method publishes concept-example relevance. `ConceptSelection` turns it and the labels into a named TensorDict value, and `ChannelConditionedLRP` reads that value during the second pass. No callback or prepared-hook state crosses the boundary.

In [ ]:
from tdhook.attribution import LRP
from tdhook.concepts import ChannelConditionedLRP, ConceptSelection
from tdhook.workflow import Workflow

concept_workflow = Workflow(
    LRP(
        input_modules=["linear1"],
        attribution_key=("attributions", "concept_examples"),
        warn_on_missing_rule=False,
    ),
    ConceptSelection(("attributions", "concept_examples", "linear1")),
    ChannelConditionedLRP(LRP(warn_on_missing_rule=False), condition_module="linear1"),
)
concept_artifacts = TensorDict({"input": examples, "concept_labels": labels}, batch_size=[len(examples)])
concept_plan = concept_workflow.plan(concept_model, concept_artifacts)
assert concept_plan.model_passes == 2
[(execution.steps, execution.model_passes) for execution in concept_plan.executions]

In [ ]:
concept_result = concept_workflow(concept_model, concept_artifacts)
selection = concept_result[("metrics", "concept_selection")]
conditioned = concept_result[("attributions", "conditioned", "input")]
print("selected channel:", selection["channel"][0].item())
print("conditioned relevance shape:", tuple(conditioned.shape))

## Conditioned intrinsic dimension: one model pass

Only activation capture runs the model. Channel selection, TwoNN estimation, and summary are ordinary TensorDict operators. Plotting or domain rendering belongs downstream of these stable artifacts.

In [ ]:
from tdhook.dimension import channel_conditioned_samples, conditioned_dimension_workflow
from tdhook.latent import ActivationCaching
from tdhook.latent.dimension_estimation import TwoNnDimensionEstimator

dimension_workflow = conditioned_dimension_workflow(
    ActivationCaching("features", cache_key=("activations", "cache")),
    "features",
    channel_conditioned_samples,
    TwoNnDimensionEstimator(),
)
dimension_artifacts = TensorDict({"input": examples}, batch_size=[len(examples)])
dimension_plan = dimension_workflow.plan(dimension_model, dimension_artifacts)
assert dimension_plan.model_passes == 1
[(execution.steps, execution.model_passes, execution.coexecuted) for execution in dimension_plan.executions]

In [ ]:
dimension_result = dimension_workflow(dimension_model, dimension_artifacts)
samples = dimension_result[("activations", "samples")].data
dimensions = dimension_result[("metrics", "dimension")].data
summary = dimension_result[("metrics", "dimension_summary")].data
print("samples:", tuple(samples.shape), "dimensions:", tuple(dimensions.shape))
summary